# 21-08 · Полная игра целиком

Практика к разделу [«Полный код»](../../site/chapters/glava-21/21-08-polnyj-kod-itogi.html). Полный файл — `projects/pygame/space-shooter/space_shooter.py`.

## Цель

Прогнать игру много кадров подряд и убедиться, что движение корабля, появление врагов, отрисовка и завершение игры работают вместе, как единое целое.

## Про игровой цикл в этом ноутбуке

В обычном `.py`-файле игровой цикл — это `while rabotaet:`, зависящий от событий пользователя (закрытие окна, нажатия клавиш). В автоматически выполняемом ноутбуке некому создавать такие события, поэтому здесь мы прогоняем фиксированное число кадров через `for kadr in range(N):` — логика каждого отдельного кадра (движение, столкновения, отрисовка, `clock.tick()`) при этом точно такая же, как в настоящей игре из `projects/pygame/space-shooter/space_shooter.py`.

In [ ]:
import random

import pygame

SHIRINA, VYSOTA = 480, 720
FPS = 60

KORABL_SHIRINA, KORABL_VYSOTA = 44, 44
KORABL_SKOROST = 6

PULYA_SHIRINA, PULYA_VYSOTA = 6, 18
PULYA_SKOROST = 9

VRAG_SHIRINA, VRAG_VYSOTA = 32, 28
VRAG_SKOROST = 2
INTERVAL_POYAVLENIYA_VRAGA = 45

BELYJ = (255, 255, 255)
CHERNYJ = (10, 10, 20)
ZELYONYJ = (80, 220, 120)
KRASNYJ = (230, 60, 60)
ZHYOLTYJ = (240, 220, 80)

pygame.init()
screen = pygame.display.set_mode((SHIRINA, VYSOTA))
pygame.display.set_caption("Космический шутер")
clock = pygame.time.Clock()
shrift = pygame.font.SysFont(None, 32)
shrift_bolshoj = pygame.font.SysFont(None, 64)


def novaya_igra():
    return {
        "korabl": pygame.Rect(
            SHIRINA // 2 - KORABL_SHIRINA // 2,
            VYSOTA - KORABL_VYSOTA - 20,
            KORABL_SHIRINA,
            KORABL_VYSOTA,
        ),
        "puli": [],
        "vragi": [],
        "schet": 0,
        "kadrov_do_vraga": INTERVAL_POYAVLENIYA_VRAGA,
        "igra_okonchena": False,
    }


def obrabotat_klavishi(state, klavishi):
    korabl = state["korabl"]
    if klavishi[pygame.K_LEFT]:
        korabl.x -= KORABL_SKOROST
    if klavishi[pygame.K_RIGHT]:
        korabl.x += KORABL_SKOROST
    korabl.x = max(0, min(korabl.x, SHIRINA - KORABL_SHIRINA))


def vystrelit(state):
    korabl = state["korabl"]
    pulya = pygame.Rect(
        korabl.centerx - PULYA_SHIRINA // 2,
        korabl.top,
        PULYA_SHIRINA,
        PULYA_VYSOTA,
    )
    state["puli"].append(pulya)


def sozdat_vraga():
    x = random.randint(0, SHIRINA - VRAG_SHIRINA)
    return pygame.Rect(x, -VRAG_VYSOTA, VRAG_SHIRINA, VRAG_VYSOTA)


def obnovit_igru(state):
    if state["igra_okonchena"]:
        return

    for pulya in state["puli"]:
        pulya.y -= PULYA_SKOROST
    state["puli"] = [p for p in state["puli"] if p.bottom > 0]

    state["kadrov_do_vraga"] -= 1
    if state["kadrov_do_vraga"] <= 0:
        state["vragi"].append(sozdat_vraga())
        state["kadrov_do_vraga"] = INTERVAL_POYAVLENIYA_VRAGA

    for vrag in state["vragi"]:
        vrag.y += VRAG_SKOROST

    novye_puli = []
    novye_vragi = list(state["vragi"])
    for pulya in state["puli"]:
        popala = False
        for vrag in list(novye_vragi):
            if pulya.colliderect(vrag):
                novye_vragi.remove(vrag)
                state["schet"] += 10
                popala = True
                break
        if not popala:
            novye_puli.append(pulya)
    state["puli"] = novye_puli
    state["vragi"] = novye_vragi

    for vrag in state["vragi"]:
        if vrag.bottom >= VYSOTA or vrag.colliderect(state["korabl"]):
            state["igra_okonchena"] = True
            break


def narisovat(state):
    screen.fill(CHERNYJ)
    pygame.draw.rect(screen, ZELYONYJ, state["korabl"])
    for pulya in state["puli"]:
        pygame.draw.rect(screen, ZHYOLTYJ, pulya)
    for vrag in state["vragi"]:
        pygame.draw.rect(screen, KRASNYJ, vrag)

    tablo = shrift.render(f"Счёт: {state['schet']}", True, BELYJ)
    screen.blit(tablo, (10, 10))

    if state["igra_okonchena"]:
        nadpis = shrift_bolshoj.render("ИГРА ОКОНЧЕНА", True, BELYJ)
        rect = nadpis.get_rect(center=(SHIRINA // 2, VYSOTA // 2))
        screen.blit(nadpis, rect)

    pygame.display.flip()

## Полная симуляция — движение и появление врагов без стрельбы

Чтобы результат был предсказуемым (враги не уничтожаются случайными попаданиями), в этом прогоне корабль просто двигается, а первый же враг долетает до низа экрана и завершает игру — ровно так, как описано в разделе «Уничтожаем космический корабль!».

In [ ]:
state = novaya_igra()
random.seed(3)

# При VRAG_SKOROST = 2 px/кадр первому врагу нужно больше 400/2 = 200 кадров,
# только чтобы пересечь VYSOTA = 720 px по вертикали — 500 кадров даёт запас.
for kadr in range(500):
    klavishi = {
        pygame.K_LEFT: kadr % 20 < 10,
        pygame.K_RIGHT: kadr % 20 >= 10,
    }
    obrabotat_klavishi(state, klavishi)
    obnovit_igru(state)
    narisovat(state)
    clock.tick(FPS)
    if state["igra_okonchena"]:
        break

print("Кадров прогнано:", kadr + 1)
print("Финальный счёт:", state["schet"])
print("Игра окончена:", state["igra_okonchena"])

## Проверка результата

In [ ]:
assert state["igra_okonchena"] is True, "за 500 кадров хотя бы один враг должен долететь до низа экрана"
assert kadr + 1 <= 500
print(f"Верно: игра завершилась на кадре {kadr + 1} со счётом {state['schet']}.")
pygame.quit()